# 00 - Getting Started with the QICK Emulator

**Objective:** Verify your QICK + emulator installation, initialize the emulator in place of a physical RFSoC board, and understand the basic hardware configuration.

This is a 1:1 port of the on-hardware tutorial `00_Getting_Started.ipynb` (mirrored under [`../../docs/source/tutorials/`](../../docs/source/tutorials/) in this repo), running against the Verilator emulator instead of a physical board. Sections and their numbering match the original; only the board-connection and hardware-verification steps are adapted, since the emulator's record-then-replay architecture has no live-register-read equivalent (see Section 4).

```python
soc = QickEmu(str(CFG_PATH))     # instead of QickSoc(BITSTREAM_PATH)
soc = soc.prepare_emu(memdir=OUT)
iq_list = prog.acquire_decimated(soc)   # or prog.acquire(...)
```

Before running this notebook, make sure your environment is set up per [`emulator/README.md`](../../README.md) (Verilator 5.042, git submodules, the `qick-venv` kernel).

## 1. Setup and Imports

In [ ]:
# Jupyter notebook setup
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import sys
import pathlib
import logging

# Local qick_lib / emulator path setup (see emulator/notebooks/00_intro_emu.ipynb)
REPO_ROOT = pathlib.Path.cwd().parent.parent   # emulator/ (tutorial/ nests one level deeper)
sys.path.insert(0, str(REPO_ROOT / '..' / 'qick_lib'))
sys.path.insert(0, str(REPO_ROOT / 'software'))

# Core QICK imports
import qick
from qick import *

# tProc v2 specific classes
from qick.asm_v2 import AveragerProgramV2, QickSweep1D, QickSpan

# Emulator entry point (drop-in replacement for QickSoc)
from qick_emu import QickEmu

# Optional: increase logging level for debugging
# logging.basicConfig(level=logging.DEBUG)
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')
# Reduce noise from the tProc core
logging.getLogger("qick_processor").setLevel(logging.WARNING)

# Display the loaded QICK library version
print(f"QICK Version: {qick.get_version()}")

**Explanation:** Same imports as the on-hardware notebook, plus `QickEmu` from `emulator/software/qick_emu.py` — the emulator's drop-in replacement for `QickSoc`. The `qick.asm_v2` module contains the classes for building tProc v2 programs; that part of your code doesn't change between hardware and emulator.

## 2. Initialize the QICK Emulator

On real hardware you'd point `QickSoc` at a firmware `.bit` file specific to your board. The emulator instead loads a board config JSON that describes the same firmware (channels, clocks, tProc) as a Verilator simulation — no board, no bitstream file, no Xilinx tools required.

In [ ]:
# --- No firmware .bit file or physical board needed ---
# On real hardware you'd do:
#     BITSTREAM_PATH = '/path/to/your/firmware.bit'
#     soc = QickSoc(BITSTREAM_PATH)
#
# The emulator loads the same kind of information from a JSON config instead,
# and hands back a `soc`-compatible object with the same API used below.
CFG_PATH = REPO_ROOT / 'config' / 'qick_emu_config.json'

soc    = QickEmu(str(CFG_PATH))
soccfg = soc.soccfg  # 'soccfg' is a common alias used throughout QICK examples

# Where this notebook's simulation artifacts (CSVs, .mem files) get written
ARTIFACTS_ROOT = REPO_ROOT / 'artifacts' / '00_getting_started'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print("\n=== Emulator Initialized ===")

## 3. Inspect the Hardware Configuration

The `soccfg` object is a `QickConfig` — the exact same class used on real hardware — so it contains the same information about the (simulated) firmware: clock frequencies, generator channels, readout channels, digital I/O, and tProc capabilities.

In [ ]:
print(soccfg)

**Explanation:** The printed output shows:
- **Global clocks:** tProc timing clock, RF reference frequency
- **Signal generators:** Number of channels, sample rates, envelope memory size
- **Readout channels:** Sample rates, buffer sizes, trigger mappings
- **tProc configuration:** Program memory size, data memory size, waveform memory size, core clock

This is the same information — and the same object type — you'd inspect on real hardware. `qick_emu_config.json` does not model DDR4 buffers (the emulator testbench doesn't instantiate that IP), so you won't see a `DDR4 memory buffer` line here even if the board config it's based on has one.

## 4. Quick Emulator Test

On hardware, `get_tproc_counter()` and `read_mem()` read live registers over the PYNQ/AXI interface to confirm the board is responding. The emulator has no live board to poll: it works by **recording** the AXI writes your program issues and then **replaying** them in a single batched Verilator simulation, so there's nothing to read back until a simulation has actually run (`soc.reg_read()` is a documented no-op for exactly this reason).

The emulator-equivalent smoke test is to run one minimal program through the full pipeline — Python → AXI record → Verilator → CSV — and confirm real data comes back. That's what actually proves your toolchain (Verilator 5.042, the git submodules, the `qick-venv` environment) is working end-to-end.

In [ ]:
class SmokeTestProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_pulse(ch=gen_ch, name='mypulse', ro_ch=ro_ch,
                       style='const', length=0.1,
                       freq=cfg['freq'], phase=0, gain=1.0)

        self.add_readoutconfig(ch=ro_ch, name='myro', freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name='myro', t=0)

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name='mypulse', t=0)


config = {
    'gen_ch':    0,     # QICKEmu channel assignment, see 00_intro_emu.ipynb
    'ro_ch':     0,
    'freq':      100,   # MHz
    'trig_time': 0.40,  # us
    'ro_len':    0.3,   # us
}

prog = SmokeTestProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

OUT = ARTIFACTS_ROOT / 'smoke_test'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Artifacts -> {OUT}')

# First simulation of the kernel session: build=True runs `make verilate`,
# a full compile of the Verilator testbench (the slow step, ~1-2 min).
# It only needs to happen once per session -- see the PERFORMANCE NOTE in
# 00_intro_emu.ipynb for the build=False pattern used on later runs.
soc = soc.prepare_emu(memdir=OUT, build=True, verbose=True)

iq_list = prog.acquire_decimated(soc, rounds=1)
t = prog.get_time_axis(ro_index=0)

print(f"\n✓ Emulator pipeline verified: acquired {iq_list[0].shape[0]} decimated I/Q samples.")

plt.plot(t, iq_list[0][:, 0], label="I value")
plt.plot(t, iq_list[0][:, 1], label="Q value")
plt.legend()
plt.ylabel("a.u.")
plt.xlabel("us");

**Explanation:** `prepare_emu()` resets the emulator's recorder and points it at a fresh artifacts directory; `acquire_decimated()` then records the program's AXI transactions, invokes `make verilate` + the compiled testbench binary, and parses the resulting `dec_out_ch0.csv` back into the `iq_list` array you'd get on real hardware. Getting a nonzero-length I/Q array back — and a pulse-shaped plot above — confirms the whole chain works.

## 5. Summary

You are now ready to start writing QICK experiments against the emulator. [`00_intro_emu.ipynb`](../00_intro_emu.ipynb) picks up from here with a full set of pulse-sequencing and acquisition examples (the emulator port of the original `00_intro.ipynb`).

**Key takeaways:**
- Use `QickEmu(cfg_path)` instead of `QickSoc(bitfile_path)` — no board or bitstream needed
- The `soccfg` object holds the same hardware configuration information as on real hardware
- `prog.acquire()` / `prog.acquire_decimated()` transparently dispatch to the emulator when `soc` is a `QickEmu`
- The emulator has no live register reads (`reg_read()` is a no-op); to verify the toolchain, run a minimal program end-to-end instead
- `build=True` (compiling the Verilator testbench) only needs to happen once per kernel session — see the performance note in `00_intro_emu.ipynb`